# ICML 2027 Week 1: Phase 0 Diagnostic & Pilot Sequential ELBO Rewriting

**What this notebook runs (10–15 mins on 1x Colab A100 / L4):**
1. Checks GPU & installs dependencies.
2. Unpacks `agent-sft-align.zip` (or clones your private GitHub repo) and installs `agent-sft-align`.
3. Automatically pulls expert **ALFWorld** (or **WebShop** / **ScienceWorld**) trajectories from `agent-eto/eto-sft-trajectory`.
4. Runs **Phase 0 Diagnostic + Pilot ELBO Rewriting**:
   - Quantifies the **Off-Policy Perplexity Gap** between GPT-4 Teacher `Thought` ($r_t^*$) and Verified `Action` ($a_t^*$).
   - Detects environment no-op / looping steps (`Nothing happens.`) for **Causal-Preserving Step Masking**.
   - Rewrites pilot trajectories with **Sequential ELBO + CVaR Anti-Leakage Selection** and renders **Figure 1a & 1b** inline.
5. Packages `phase0_results.zip` for download so you can drop it straight back into the local repo for analysis!

In [ ]:
!nvidia-smi
!pip install -q "transformers>=4.45.0" "accelerate>=0.33.0" "huggingface_hub>=0.23.0" "peft>=0.12.0" pyyaml tqdm matplotlib

In [ ]:
# Option A: Upload `agent-sft-align.zip` to the Colab sidebar (/content/agent-sft-align.zip)
# Option B: Or set REPO_GIT_URL to your private GitHub repo!
import os

REPO_GIT_URL = ""  # e.g. "https://<TOKEN>@github.com/<USER>/agent-sft-align.git"

if REPO_GIT_URL:
    !rm -rf /content/agent-sft-align && git clone {REPO_GIT_URL} /content/agent-sft-align
elif os.path.exists("/content/agent-sft-align.zip"):
    !rm -rf /content/agent-sft-align && unzip -q /content/agent-sft-align.zip -d /content/
else:
    raise FileNotFoundError("Please upload `agent-sft-align.zip` to /content/ or set REPO_GIT_URL above.")

%cd /content/agent-sft-align
!pip install -q -e .

In [ ]:
# Configure Target Base/Instruct Model & Task
# On A100 (40GB/80GB): `Qwen/Qwen2.5-7B-Instruct` or `Qwen/Qwen2.5-3B-Instruct` both run smoothly!
# On L4 (24GB): `Qwen/Qwen2.5-3B-Instruct` is super fast (~8 mins), `7B` also fits in bf16.
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
TASK = "alfworld"  # choices: alfworld, webshop, sciworld

!python3 scripts/run_phase0_diagnostic.py \
  --model "{MODEL_ID}" \
  --task "{TASK}" \
  --num-diagnostic-trajectories 40 \
  --num-rewrite-trajectories 8 \
  --num-candidates 6 \
  --lambda-prior 0.5 \
  --gamma-cvar 0.15 \
  --output-dir "outputs/phase0_{TASK}"

In [ ]:
# Inspect Inline Figures & Qualitative Rewrites
import json
from IPython.display import Image, display

out_dir = f"outputs/phase0_{TASK}"
display(Image(filename=f"{out_dir}/fig1a_prior_gap_nll.png"))
display(Image(filename=f"{out_dir}/fig1b_sufficiency_vs_prior.png"))

with open(f"{out_dir}/qualitative_samples.json", "r", encoding="utf-8") as f:
    samples = json.load(f)

for s in samples[:4]:
    print("=" * 80)
    print(f"[{s['trajectory_id']} | Step {s['step_idx']}] Action: {s['action']}")
    print(f"  Teacher (PPL={s['teacher_ppl']}, InfoGain={s['teacher_info_gain']}): {s['teacher_thought']}")
    print(f"  Ours    (PPL={s['rewritten_ppl']}, InfoGain={s['rewritten_info_gain']}): {s['rewritten_thought']}")

In [ ]:
# Zip & Download Results to Send Back for Local Analysis!
from google.colab import files

!zip -r /content/phase0_results.zip outputs/
files.download("/content/phase0_results.zip")